In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

# 1. Load dataset
df = pd.read_csv('earthquakes.csv')
df_processed = df.copy()

In [ ]:


# 2. Missing Value Strategy (Median / Mode)
numerical_cols = df_processed.select_dtypes(include=[np.number]).columns
categorical_cols = df_processed.select_dtypes(exclude=[np.number]).columns

for col in numerical_cols:
    df_processed[col] = df_processed[col].fillna(df_processed[col].median())
for col in categorical_cols:
    df_processed[col] = df_processed[col].fillna(df_processed[col].mode())


In [ ]:
# 3. Handle outliers using IQR (Capping)
def handle_outliers_iqr(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    data[col] = np.where(data[col] < lower_bound, lower_bound, data[col])
    data[col] = np.where(data[col] > upper_bound, upper_bound, data[col])

features_for_outliers = ['impact.gap', 'impact.magnitude', 'location.depth', 'location.distance']
for col in features_for_outliers:
    handle_outliers_iqr(df_processed, col)

In [ ]:
# 4. Normalize numerical features
scaler_minmax = MinMaxScaler()
scaler_standard = StandardScaler()

df_processed['impact.magnitude_minmax'] = scaler_minmax.fit_transform(df_processed[['impact.magnitude']])
df_processed['location.depth_zscore'] = scaler_standard.fit_transform(df_processed[['location.depth']])

In [ ]:
# 5. PCA on Highly Correlated Features
corr_val = df_processed['impact.magnitude'].corr(df_processed['impact.significance'])

if abs(corr_val) > 0.7:
    pca = PCA(n_components=1)
    df_processed['pca_magnitude_significance'] = pca.fit_transform(df_processed[['impact.magnitude', 'impact.significance']])
    # Drop the original redundant columns
    df_processed.drop(columns=['impact.magnitude', 'impact.significance'], inplace=True)

In [ ]:
# Clean up redundant/constant columns
df_processed.drop(columns=['time.year'], inplace=True, errors='ignore')

# Save processed dataset
df_processed.to_csv('earthquakes_processed.csv', index=False)